## Dependencies and Imports

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q -U accelerate==0.27.2
!pip install -q -U transformers==4.37.2
!pip install -q -U datasets==2.17.0
!pip install -q -U evaluate==0.4.1

In [ ]:
import numpy as np
import pandas as pd
import ast
from datasets import Dataset, DatasetDict

import torch
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.optim import AdamW

from transformers import AutoTokenizer, DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification
from transformers import get_scheduler, set_seed

from sklearn.metrics import f1_score, hamming_loss, recall_score, precision_score
from huggingface_hub import Repository, get_full_repo_name
from huggingface_hub import create_repo

import time
import math
import copy
from tqdm.auto import tqdm

seed = 17 # 17, 987, 1153
set_seed(seed)

## Necessary Functions

In [ ]:
def preprocess_function(example):
    text = example['text']
    related_labels = ast.literal_eval(example['related_labels'])
    # a list of empty labels
    temp_labels = [0. for i in range(len(labels))]

    for rl in related_labels:
        label_id = label2id[rl]
        temp_labels[label_id] = 1.

    example = tokenizer(text, padding='max_length', truncation=True)
    example['labels'] = temp_labels
    return example

#get number of model parameters
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"trainable model parameters: {trainable_model_params/1e9:.3f} B\nall model parameters: {all_model_params/1e9:.3f} B\npercentage of trainable model parameters: {100 * trainable_model_params / all_model_params:.2f}%"

In [ ]:
# Multi-Label Classification Evaluation Metrics
def multi_labels_metrics(predictions, label_values, threshold=0.5):
    sigmoid = torch.nn.Sigmoid()
    # apply sigmoid on predictions to convert values between 0 and 1
    probs = sigmoid(torch.Tensor(predictions))

    # using threshold to turn them into integer predictions
    y_pred = np.zeros(probs.shape)
    y_pred[np.where(probs>=threshold)] = 1
    y_true = label_values

    f1 = f1_score(y_true, y_pred, average = 'samples')
    precision = precision_score(y_true, y_pred, average = 'samples')
    recall = recall_score(y_true, y_pred, average = 'samples')
    hamming = hamming_loss(y_true, y_pred)

    metrics = {
      "hamming_loss": hamming,
      "f1": f1,
      "recall": recall,
      "precision": precision
    }

    return metrics

## Load Dataset and Tokenization

In [ ]:
model_path = 'emilyalsentzer/Bio_Discharge_Summary_BERT'

tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast= True)

if tokenizer.model_max_length > 512:
    print(f"Max model length was {tokenizer.model_max_length}")
    tokenizer.model_max_length = 512
    print(f"Max model length is changed to 512.")
else:
    print('Pre-defined max model length is', tokenizer.model_max_length)

In [ ]:
# set to False when you want to train on non-modified train data
use_train_data_with_modifications = True

# get validation data
valid_data = pd.read_csv('/content/drive/MyDrive/Differential Diagnosis Project/Dataset/Sampled DDXPlus for Classification/val/validation_set.csv')

if use_train_data_with_modifications:
  file_paths = ['/content/drive/MyDrive/Differential Diagnosis Project/Dataset/Sampled DDXPlus for Classification/train/with_modifications/train_set_unchanged.csv',
                '/content/drive/MyDrive/Differential Diagnosis Project/Dataset/Sampled DDXPlus for Classification/train/with_modifications/train_set_with_med_term_diversity.csv',
                '/content/drive/MyDrive/Differential Diagnosis Project/Dataset/Sampled DDXPlus for Classification/train/with_modifications/train_set_paraphrased.csv']
  col_names = list(valid_data.columns)
  train_data = pd.DataFrame(columns = col_names)

  for path in file_paths:
    temp_df = pd.read_csv(path)
    train_data  = pd.concat([train_data , temp_df] , axis=0, ignore_index=True)

  print('Selected training data contains some modifications.')

else:
  train_data = pd.read_csv('/content/drive/MyDrive/Differential Diagnosis Project/Dataset/Sampled DDXPlus for Classification/train/train_set.csv')
  print('Selected training data contains no modifications.')

In [ ]:
# Sanity Check
print(train_data['text'][45001])

In [ ]:
# dataset dictionary
dataset = DatasetDict()
train_dataset = Dataset.from_pandas(train_data)
valid_dataset = Dataset.from_pandas(valid_data)
dataset['train'] = train_dataset
dataset['test'] = valid_dataset
print('\n', dataset)

# labels and label_ids
labels = ast.literal_eval(train_data['all_labels'][0])
label2id = {label:idx for idx, label in enumerate(labels)}
id2label = {idx:label for label, idx in label2id.items()}

# dataset tokenization
tokenized_dataset = dataset.map(preprocess_function, remove_columns=dataset["train"].column_names)
columns = tokenized_dataset['train'].column_names
tokenized_dataset.set_format(type='torch', columns=columns)

# Create Dataloaders
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

batch_size = 16

train_dataloader = DataLoader(
       tokenized_dataset["train"], shuffle=True, batch_size=batch_size, collate_fn=data_collator
    )
eval_dataloader = DataLoader(
       tokenized_dataset["test"], shuffle=True, batch_size=batch_size, collate_fn=data_collator
    )

## Load Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=len(labels),
                                                           id2label=id2label, label2id=label2id,
                                                           problem_type = "multi_label_classification")

In [ ]:
print(print_number_of_trainable_model_parameters(model))

## Training Parameters

In [ ]:
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
num_epochs = 10
num_training_steps = num_epochs * len(train_dataloader)
threshold = 0.5

lr_scheduler = get_scheduler(
    "constant",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
print('number of training steps: ',num_training_steps)

## HF Repo Information

In [ ]:
model_name = "BioDisSumBert_DDXPlus_2"
repo_name = get_full_repo_name(model_name)
print(repo_name)

# comment this line if the repo is already created
create_repo(repo_name, private=True)

# clone repo to local directory
output_dir = model_name
repo = Repository(output_dir, clone_from=repo_name)

## Training Loop

In [ ]:
since = time.time()
writer = SummaryWriter(log_dir= output_dir + '/runs/' + model_name)

best_eval_loss = float('inf')
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
best_model_wts = copy.deepcopy(model.state_dict()) #best weights
print('Device: ', device)

for epoch in range(num_epochs):
    print(f'Epoch {epoch}/{num_epochs - 1}')
    print('-' * 10)

    # Initialize variables to accumulate training and validation loss
    epoch_train_loss = 0.0
    epoch_eval_loss = 0.0

    #Training
    model.train()
    for batch in tqdm(train_dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        epoch_train_loss += loss.detach().float()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    epoch_train_loss = epoch_train_loss.item() / len(train_dataloader)
    print(f">>> Epoch {epoch}: Training Loss: {epoch_train_loss}")

    writer.add_scalar("Train Loss", epoch_train_loss, epoch)

    #Evaluation
    all_preds = torch.tensor([]).to(device)
    all_labels = torch.tensor([]).to(device)
    model.eval()
    for batch in tqdm(eval_dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
            preds = outputs.logits.squeeze()

        loss = outputs.loss
        epoch_eval_loss += loss.detach().float()
        all_preds = torch.cat((all_preds, preds))
        all_labels = torch.cat((all_labels, batch['labels']))

    # Calculate average validation loss for the epoch
    epoch_eval_loss = epoch_eval_loss.item() / len(eval_dataloader)
    metric_scores = multi_labels_metrics(predictions=all_preds.cpu(), label_values=all_labels.cpu(), threshold=threshold)

    if epoch_eval_loss < best_eval_loss:
        best_eval_loss = epoch_eval_loss
        best_model_wts = copy.deepcopy(model.state_dict()) #best weights
        # save model to hub
        model.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)
        repo.push_to_hub()

    print(f">>> Validation Epoch {epoch}: Loss - {epoch_eval_loss}, F1 - {metric_scores['f1']}, Precision - {metric_scores['precision']}, Recall - {metric_scores['recall']}, Hamming_Loss - {metric_scores['hamming_loss']}")

    writer.add_scalar("Eval Loss", epoch_eval_loss, epoch)
    writer.add_scalar("Eval F1", metric_scores["f1"], epoch)
    writer.add_scalar("Eval Precision", metric_scores["precision"], epoch)
    writer.add_scalar("Eval Recall", metric_scores["recall"], epoch)
    writer.add_scalar("Eval Hamming_Loss", metric_scores["hamming_loss"], epoch)

    time_elapsed = time.time() - since
    print(f'Epoch complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print()

writer.flush()
writer.close()

time_elapsed = time.time() - since
print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
print(f'Best eval loss: {best_eval_loss:.2f}')
# load best model weights
model.load_state_dict(best_model_wts)

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
repo.push_to_hub()

In [ ]:
from google.colab import runtime
runtime.unassign()